In [1]:

!pip -q install pandas scikit-learn textblob ipywidgets joblib

import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from textblob import TextBlob
from IPython.display import display
import ipywidgets as widgets

print("Day 1 libraries loaded successfully!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.3 MB/s eta 0:00:00
Day 1 libraries loaded successfully!


In [2]:

# Sample app usage dataset
data = {
    "login_days": [20, 3, 15, 2, 25, 7, 18, 4, 22, 5],
    "session_minutes": [45, 10, 35, 8, 60, 15, 40, 12, 55, 9],
    "features_used": [8, 2, 6, 1, 9, 3, 7, 2, 8, 2],
    "notifications_opened": [15, 1, 10, 0, 20, 3, 12, 1, 18, 2],
    "retained": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

feature_columns = [
    "login_days",
    "session_minutes",
    "features_used",
    "notifications_opened"
]

X = df[feature_columns]
y = df["retained"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Dataset created successfully!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Test accuracy:", round(accuracy * 100, 2), "%")

# Predict a new user's retention
new_user = pd.DataFrame([{
    "login_days": 12,
    "session_minutes": 30,
    "features_used": 5,
    "notifications_opened": 8
}], columns=feature_columns)

prediction = model.predict(new_user)[0]
probabilities = model.predict_proba(new_user)[0]

print("\n===== RETENTION ANALYSIS =====")

if prediction == 1:
    print("Prediction: User is likely to be retained")
else:
    print("Prediction: User may be at risk of churn")

print(
    "Estimated retention probability:",
    round(probabilities[list(model.classes_).index(1)] * 100, 2),
    "%"
)

# Save the trained model
joblib.dump(model, "retention_model.pkl")

print("\nRetention model saved successfully!")


Dataset created successfully!
Training samples: 7
Testing samples: 3
Test accuracy: 100.0 %

===== RETENTION ANALYSIS =====
Prediction: User is likely to be retained
Estimated retention probability: 88.0 %

Retention model saved successfully!


In [3]:

def analyze_mood(previous_entries, current_entry):
    if not current_entry.strip():
        print("Please enter a journal entry.")
        return

    # Sentiment scores
    previous_score = (
        TextBlob(previous_entries).sentiment.polarity
        if previous_entries.strip()
        else 0.0
    )

    current_score = TextBlob(current_entry).sentiment.polarity

    # Sentiment classification
    if current_score > 0.1:
        sentiment = "Positive"
    elif current_score < -0.1:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    # Detect a significant negative shift
    negative_shift = current_score < previous_score - 0.2

    # Basic safety keyword screening
    safety_keywords = [
        "suicide",
        "kill myself",
        "end my life",
        "self harm",
        "self-harm",
        "hurt myself"
    ]

    text = current_entry.lower()
    safety_flag = any(word in text for word in safety_keywords)

    # Display results
    print("===== MOOD JOURNAL ANALYSIS =====")
    print("Current Sentiment:", sentiment)
    print("Current Sentiment Score:", round(current_score, 2))
    print("Previous Sentiment Score:", round(previous_score, 2))

    if negative_shift:
        print("Mood Trend: Negative shift detected")
    else:
        print("Mood Trend: No significant negative shift detected")

    if safety_flag:
        print("\nSafety Status: A safety-related phrase was detected.")
        print(
            "Automated analysis cannot assess immediate safety. "
            "Please seek support from a trusted adult or qualified professional."
        )
    else:
        print("\nSafety Status: No listed safety keywords detected.")

    print("\nNote: Sentiment scores and keyword checks are limited.")
    print("This tool does not diagnose mental-health conditions.")


# Sample journal history
previous_entries = """
I had a great day at college.
I enjoyed spending time with my friends.
"""

current_entry = """
Today was difficult and I felt frustrated.
Nothing seemed enjoyable.
"""

analyze_mood(previous_entries, current_entry)


===== MOOD JOURNAL ANALYSIS =====
Current Sentiment: Negative
Current Sentiment Score: -0.23
Previous Sentiment Score: 0.65
Mood Trend: Negative shift detected

Safety Status: No listed safety keywords detected.

Note: Sentiment scores and keyword checks are limited.
This tool does not diagnose mental-health conditions.


In [4]:

previous_box = widgets.Textarea(
    value=previous_entries,
    description="Previous:",
    placeholder="Enter previous journal entries",
    layout=widgets.Layout(width="90%", height="100px")
)

current_box = widgets.Textarea(
    description="Current:",
    placeholder="Write your current journal entry here...",
    layout=widgets.Layout(width="90%", height="120px")
)

mood_button = widgets.Button(
    description="Analyze Mood",
    button_style="primary"
)

mood_output = widgets.Output()

def run_mood_analysis(button):
    with mood_output:
        mood_output.clear_output(wait=True)

        analyze_mood(
            previous_box.value,
            current_box.value
        )

mood_button.on_click(run_mood_analysis)

display(previous_box, current_box, mood_button, mood_output)


Textarea(value='\nI had a great day at college.\nI enjoyed spending time with my friends.\n', description='Pre…

Textarea(value='', description='Current:', layout=Layout(height='120px', width='90%'), placeholder='Write your…

Button(button_style='primary', description='Analyze Mood', style=ButtonStyle())

Output()

In [5]:

login_days_box = widgets.IntText(
    value=12, description="Login Days:"
)

session_box = widgets.IntText(
    value=30, description="Session Minutes:"
)

features_box = widgets.IntText(
    value=5, description="Features Used:"
)

notifications_box = widgets.IntText(
    value=8, description="Notifications:"
)

retention_button = widgets.Button(
    description="Predict Retention",
    button_style="primary"
)

retention_output = widgets.Output()

def run_retention_prediction(button):
    with retention_output:
        retention_output.clear_output(wait=True)

        values = [
            login_days_box.value,
            session_box.value,
            features_box.value,
            notifications_box.value
        ]

        if any(value < 0 for value in values):
            print("Please enter non-negative values.")
            return

        user_data = pd.DataFrame(
            [{
                "login_days": values[0],
                "session_minutes": values[1],
                "features_used": values[2],
                "notifications_opened": values[3]
            }],
            columns=feature_columns
        )

        result = model.predict(user_data)[0]

        print("===== RETENTION PREDICTION =====")

        if result == 1:
            print("User is likely to be retained.")
        else:
            print("User may be at risk of churn.")

retention_button.on_click(run_retention_prediction)

display(
    login_days_box,
    session_box,
    features_box,
    notifications_box,
    retention_button,
    retention_output
)


IntText(value=12, description='Login Days:')

IntText(value=30, description='Session Minutes:')

IntText(value=5, description='Features Used:')

IntText(value=8, description='Notifications:')

Button(button_style='primary', description='Predict Retention', style=ButtonStyle())

Output()

In [6]:

from textblob import TextBlob

previous_entries = """
I had a great day at college.
I enjoyed spending time with my friends.
"""

current_entry = """
Today was difficult and I felt frustrated.
Nothing seemed enjoyable.
"""

previous_score = TextBlob(previous_entries).sentiment.polarity
current_score = TextBlob(current_entry).sentiment.polarity

print("Previous Score:", round(previous_score, 2))
print("Current Score:", round(current_score, 2))

if current_score < previous_score - 0.2:
    print("Negative shift detected")
else:
    print("No significant negative shift detected")


Previous Score: 0.65
Current Score: -0.23
Negative shift detected


In [7]:

def safety_check(journal):
    safety_keywords = [
        "suicide",
        "kill myself",
        "end my life",
        "self-harm",
        "self harm",
        "hurt myself"
    ]

    text = journal.lower()

    for keyword in safety_keywords:
        if keyword in text:
            print("Safety alert: Please seek support from a trusted person.")
            return

    print("No safety keywords detected.")


test_journal = "I feel very sad today."
safety_check(test_journal)


No safety keywords detected.


In [8]:

test_journal = "I am thinking about suicide."
safety_check(test_journal)


Safety alert: Please seek support from a trusted person.


In [9]:

print("===== DAY 1 FINAL TEST =====")

print("\n1. Retention Model:")
print("Model trained:", hasattr(model, "predict"))

print("\n2. Mood Analysis:")
analyze_mood(
    "I had a great day at college.",
    "Today was difficult and frustrating."
)

print("\n3. Safety Check:")
safety_check("I feel very sad today.")

print("\n===== TEST COMPLETED =====")


===== DAY 1 FINAL TEST =====

1. Retention Model:
Model trained: True

2. Mood Analysis:
===== MOOD JOURNAL ANALYSIS =====
Current Sentiment: Negative
Current Sentiment Score: -0.45
Previous Sentiment Score: 0.8
Mood Trend: Negative shift detected

Safety Status: No listed safety keywords detected.

Note: Sentiment scores and keyword checks are limited.
This tool does not diagnose mental-health conditions.

3. Safety Check:
No safety keywords detected.

===== TEST COMPLETED =====
